**รายละเอียดโจทย์การบ้าน: Hybrid Multi-Agent System**

ในโจทย์นี้ นักศึกษาจะได้ออกแบบและทดลองสร้างระบบ Multi-Agent ที่ผสมผสานความสามารถจากหลาย ๆ ตัวแทน ดังนี้

1. **ReAct (Research) Agent**

   * ใช้ฟังก์ชัน ` เพื่อดึงข้อมูลจาก Wikipedia ต`search_wikipedia(term)ามคำค้น

2. **Coding Agent**

   * ใช้ฟังก์ชัน `run_python_code(code)` เพื่อรันโค้ด Python จริง และส่งผลลัพธ์กลับมา
   * หลังรันแล้ว ให้สะท้อนผลลัพธ์ (self-reflection) เพื่อปรับปรุงโค้ดหรือคำอธิบายให้สมบูรณ์ยิ่งขึ้น
3. **Mixture-of-Agents Orchestrator**

   * ใช้ฟังก์ชัน `mixture_of_agents(user_prompt, agent_roles)`
   * กำหนดบทบาท (roles) ของ Proposer Agents อย่างน้อย 2 บทบาท (เช่น “Data Scientist”, “frontend developer”, “backend engineer”, “UX designer” เป็นต้น)
   * รวบรวม (aggregate) คำตอบจากแต่ละ Proposer แล้วสังเคราะห์ผลลัพธ์ออกมาเป็นคำตอบเดียว

> **สิ่งที่ต้องส่ง**
> * โค้ด Python ที่ประกอบด้วยการเรียกใช้ทั้ง `research_agent()`, `coding_agent()`, และ `mixture_of_agents()`


---

#### ตัวอย่างคำถามจำนวน 5 ข้อ (ในรูปแบบ Python list) สำหรับทดสอบระบบ

```python
questions = [
    # 1. หัวข้อ AI เบื้องต้น:
    "สรุปแนวคิดหลักของ 'Reinforcement Learning' จาก Wikipedia แล้วสร้างกราฟแสดง reward function ด้วย Python พร้อมสะท้อนผลลัพธ์ทุกขั้นตอน",

    # 2. วิเคราะห์ข้อมูลจริง:
    "ใช้ Research Agent ดึงข้อมูลสถิติ COVID-19 รายวันจาก Wikipedia จากนั้น Coding Agent คำนวณค่า moving average ของยอดผู้ติดเชื้อ 7 วัน และ Mixture-of-Agents ให้บทบาท 'Epidemiologist' และ 'Data Analyst' เสนอแนวทางจัดการข้อมูล",

    # 3. เปรียบเทียบอัลกอริทึม:
    "เปรียบเทียบเวลาในการรันอัลกอริทึม 'Quick Sort' และ 'Merge Sort' ด้วย Coding Agent พร้อม Self-Reflection ว่าการเลือกใช้โครงสร้างข้อมูลใดเหมาะสมกว่า และให้ Mixture-of-Agents สะท้อนมุมมองของ 'Algorithm Theorist' กับ 'Benchmark Specialist'",

    # 4. ออกแบบ Chatbot ง่าย ๆ:
    "สร้างระบบ Chatbot เบื้องต้น: Research Agent ค้นหา FAQ จาก Wikipedia, Coding Agent เขียนโค้ด Flask ให้สามารถรับข้อความและตอบกลับ, Mixture-of-Agents ให้บทบาท 'UX Designer' และ 'Backend Engineer' วิเคราะห์การทำงาน และสะท้อนปรับปรุง UX/UI",

    # 5. วิเคราะห์ชุดข้อมูล Titanic:
    "Research Agent สรุปที่มาของชุดข้อมูล Titanic จาก Wikipedia, Coding Agent โหลดข้อมูลและคำนวณอัตราการรอดชีวิตตามเพศและชั้นผู้โดยสาร พร้อม self-reflection, Mixture-of-Agents ให้บทบาท 'Statistician' และ 'Data Engineer' สรุปผลเชิงตีความ"
]
```

> แต่ละโจทย์จะต้องรันผ่านฟังก์ชัน
>
> ```python
> result = multi_agent_system(question)
> ```
>
> และ
>
> ```python
> final = mixture_of_agents(question, ["Role1", "Role2"])
> ```
>
> พร้อมแสดง output ทุกขั้นตอนและข้อคิดเห็น (reflection) ของแต่ละ agent ให้ครบถ้วน เพื่อให้เห็นภาพการทำงานร่วมกันของระบบ Multi-Agent Hybrid นี้อย่างชัดเจน


In [ ]:
import IPython
import sys

def clean_notebook():
    IPython.display.clear_output(wait=True)
    print("Notebook cleaned.")

!pip install openai wikipedia sympy requests -q

# Clean up the notebook
clean_notebook()


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

In [ ]:
import os
from openai import OpenAI
import wikipedia
from sympy import sympify
import requests
from datetime import date
import random

# Set up OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
model_name = "gpt-4o"  # Using 'gpt-4o' as a stand-in; change if needed to 'gpt-4.1' or another model

In [ ]:
import wikipedia

def search_wikipedia(term):
    """
    ค้นหาข้อมูลจาก Wikipedia ตามคำค้นที่กำหนด
    
    Args:
    term (str): คำค้นที่ต้องการค้นหาบน Wikipedia

    Returns:
    str: สรุปข้อมูลจากหน้า Wikipedia
    """
    try:
        # ใช้ฟังก์ชัน summary ของ wikipedia เพื่อดึงข้อมูลสรุปโดยไม่จำกัดจำนวนประโยค
        summary = wikipedia.summary(term)  # ไม่มีการจำกัดจำนวนประโยค
        return summary
    except wikipedia.exceptions.DisambiguationError as e:
        # ถ้าคำค้นมีหลายความหมายจะดึงรายการที่มีความหมายหลายๆ แบบ
        return f"คำค้น '{term}' มีหลายความหมาย โปรดเลือก: {e.options}"
    except wikipedia.exceptions.HTTPTimeoutError:
        return "เกิดข้อผิดพลาดในการเชื่อมต่อกับ Wikipedia"
    except wikipedia.exceptions.RedirectError:
        return "คำค้นถูกเปลี่ยนเส้นทางไปยังหน้าอื่น"
    except wikipedia.exceptions.PageError:
        return "ไม่พบข้อมูลสำหรับคำค้นนี้"
    except Exception as e:
        return f"เกิดข้อผิดพลาด: {str(e)}"

# ตัวอย่างการใช้งาน
term = "Reinforcement Learning"
result = search_wikipedia(term)
print(result)


In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_wikipedia",
            "description": "Searches Wikipedia for information based on a query.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The search query to find on Wikipedia."}
                },
                "required": ["query"]
            }
        }
    }
]

In [ ]:

available_functions = {
  "search_wikipedia":search_wikipedia
}

In [ ]:
system_prompt = "You are a helpful assistant that uses tools to answer questions."